# Data Preparation: Store 1 - GROCERY I Daily Sales

This notebook prepares a clean, leakage-safe daily time-series modeling dataset for **Store 1 / GROCERY I** daily sales from the Favorita dataset.

**Objectives**:
1. Filter and sort the target series.
2. Establish a continuous daily time index.
3. Investigate and document the modeling assumption for missing calendar dates (Christmas).
4. Preserve observed zero-sales days.
5. Engineer calendar and promotional features available at forecast origin.
6. Address multi-step forecasting leakage: distinguish features available ex-ante vs. features requiring recursive prediction.
7. Define a chronological 28-day train/test split.
8. Run strengthened multi-step leakage validation checks across the entire test horizon.
9. Save the derived modeling dataset to `data/processed_store1_grocery1.csv`.

## 1. Load and Filter Target Series

We load `train.csv` and extract the target series (`store_nbr == 1` and `family == 'GROCERY I'`), retaining `date`, `sales`, and `onpromotion`.

In [1]:
import pandas as pd
import numpy as np

# Load raw training data with parsed dates
df = pd.read_csv('../data/train.csv', parse_dates=['date'])

# Filter to Store 1, GROCERY I
target_df = (
    df[(df['store_nbr'] == 1) & (df['family'] == 'GROCERY I')][['date', 'sales', 'onpromotion']]
    .sort_values('date')
    .reset_index(drop=True)
)

print('Filtered target series shape:', target_df.shape)
target_df.head()

Filtered target series shape: (1684, 3)
Out[0]: 
        date   sales  onpromotion
0 2013-01-01     0.0            0
1 2013-01-02  2652.0            0
2 2013-01-03  2121.0            0
3 2013-01-04  2056.0            0
4 2013-01-05  2216.0            0


,date,sales,onpromotion
0,2013-01-01,0.0,0
1,2013-01-02,2652.0,0
2,2013-01-03,2121.0,0
3,2013-01-04,2056.0,0
4,2013-01-05,2216.0,0


## 2. Chronological Ordering and Date Range Inspection

We verify that the target dataframe is strictly ordered by date and check the boundaries of the time series.

In [2]:
min_date = target_df['date'].min()
max_date = target_df['date'].max()
total_recorded_days = len(target_df)

print(f"Start Date:           {min_date.strftime('%Y-%m-%d')}")
print(f"End Date:             {max_date.strftime('%Y-%m-%d')}")
print(f"Total Recorded Rows:  {total_recorded_days}")

Start Date:           2013-01-01
End Date:             2017-08-15
Total Recorded Rows:  1684


## 3. Investigation and Modeling Assumption for Missing Calendar Dates (Christmas)

Time-series lag features and rolling calculations require an unbroken daily frequency. We construct a complete daily calendar from `min_date` to `max_date` and identify absent calendar dates.

In [3]:
# Create full daily calendar
full_calendar = pd.date_range(start=min_date, end=max_date, freq='D')
missing_dates = full_calendar.difference(target_df['date'])

print(f"Expected calendar days:  {len(full_calendar)}")
print(f"Present dataset rows:    {len(target_df)}")
print(f"Missing calendar dates:  {len(missing_dates)}")

# Cross-reference missing dates with holidays_events.csv
holidays_df = pd.read_csv('../data/holidays_events.csv', parse_dates=['date'])
missing_holidays = holidays_df[holidays_df['date'].isin(missing_dates)]
missing_holidays[['date', 'type', 'locale', 'locale_name', 'description', 'transferred']]

Expected calendar days:  1688
Present dataset rows:    1684
Missing calendar dates:  4
Out[0]: 
          date     type    locale locale_name description  transferred
89  2013-12-25  Holiday  National     Ecuador     Navidad        False
155 2014-12-25  Holiday  National     Ecuador     Navidad        False
208 2015-12-25  Holiday  National     Ecuador     Navidad        False
294 2016-12-25  Holiday  National     Ecuador     Navidad        False


,date,type,locale,locale_name,description,transferred
89,2013-12-25,Holiday,National,Ecuador,Navidad,False
155,2014-12-25,Holiday,National,Ecuador,Navidad,False
208,2015-12-25,Holiday,National,Ecuador,Navidad,False
294,2016-12-25,Holiday,National,Ecuador,Navidad,False


### Modeling Assumption for Missing Dates

- **Evidence**: Exactly four dates are absent from the entire series: December 25 in 2013, 2014, 2015, and 2016. In `holidays_events.csv`, all four coincide with national `Navidad` (Christmas Day) with `transferred == False`.
- **Modeling Assumption**: The raw sales dataset does not contain an explicit closure flag. However, based on the missing dates coinciding with national Christmas Day, we adopt the practical modeling assumption that the store was closed and zero transactions occurred. We therefore set `sales = 0.0` and `onpromotion = 0` for these four dates upon reindexing.
- **Tracking**: We add an explicit indicator column `was_missing` (1 for the four Christmas dates, 0 otherwise) so downstream models and analysis can distinguish between dates absent from the raw dataset and dates present with zero sales.

In [4]:
# Reindex to full daily calendar and apply documented modeling assumption
continuous_df = (
    target_df.set_index('date')
    .reindex(full_calendar)
    .rename_axis('date')
    .reset_index()
)

# Create indicator distinguishing absent dates
continuous_df['was_missing'] = continuous_df['date'].isin(missing_dates).astype(int)

# Apply modeling assumption: sales=0.0 and onpromotion=0 for missing Christmas dates
continuous_df['sales'] = continuous_df['sales'].fillna(0.0)
continuous_df['onpromotion'] = continuous_df['onpromotion'].fillna(0).astype(int)

print('Continuous dataset shape:', continuous_df.shape)
print(f"Missing dates filled: {continuous_df['was_missing'].sum()}")
continuous_df[continuous_df['was_missing'] == 1]

Continuous dataset shape: (1688, 4)
Missing dates filled: 4
Out[0]: 
           date  sales  onpromotion  was_missing
358  2013-12-25    0.0            0            1
723  2014-12-25    0.0            0            1
1088 2015-12-25    0.0            0            1
1454 2016-12-25    0.0            0            1


,date,sales,onpromotion,was_missing
358,2013-12-25,0.0,0,1
723,2014-12-25,0.0,0,1
1088,2015-12-25,0.0,0,1
1454,2016-12-25,0.0,0,1


## 4. Preservation of Observed Zero-Sales Days

We verify that all 6 zero-sales days originally present in `train.csv` are preserved without alteration, interpolation, or conversion to NaN.

In [5]:
# Inspect observed zero-sales days originally present in train.csv
observed_zeros = continuous_df[(continuous_df['sales'] == 0.0) & (continuous_df['was_missing'] == 0)]
print(f"Total observed zero-sales days preserved: {len(observed_zeros)}")
observed_zeros[['date', 'sales', 'onpromotion', 'was_missing']]

Total observed zero-sales days preserved: 6
Out[0]: 
           date  sales  onpromotion  was_missing
0    2013-01-01    0.0            0            0
365  2014-01-01    0.0            0            0
730  2015-01-01    0.0            0            0
917  2015-07-07    0.0            0            0
1095 2016-01-01    0.0            0            0
1461 2017-01-01    0.0            0            0


,date,sales,onpromotion,was_missing
0,2013-01-01,0.0,0,0
365,2014-01-01,0.0,0,0
730,2015-01-01,0.0,0,0
917,2015-07-07,0.0,0,0
1095,2016-01-01,0.0,0,0
1461,2017-01-01,0.0,0,0


All 6 original zero-sales days (5 on January 1 New Year's Day, and 1 on July 7, 2015) remain intact as `0.0`. Combined with the 4 Christmas store closures, there are exactly 10 total zero-sales observations in the continuous series.

## 5. Calendar Features

Calendar attributes are known strictly in advance for any future forecast horizon. We extract:
- `day_of_week` (0 = Monday, ..., 6 = Sunday)
- `day_of_month` (1 – 31)
- `month` (1 – 12)
- `quarter` (1 – 4)
- `year`
- `is_weekend` (1 for Saturday and Sunday, 0 otherwise)

In [6]:
# Create calendar features
continuous_df['day_of_week'] = continuous_df['date'].dt.dayofweek
continuous_df['day_of_month'] = continuous_df['date'].dt.day
continuous_df['month'] = continuous_df['date'].dt.month
continuous_df['quarter'] = continuous_df['date'].dt.quarter
continuous_df['year'] = continuous_df['date'].dt.year
continuous_df['is_weekend'] = continuous_df['day_of_week'].isin([5, 6]).astype(int)

continuous_df[['date', 'day_of_week', 'day_of_month', 'month', 'quarter', 'year', 'is_weekend']].head()

Out[0]: 
        date  day_of_week  day_of_month  month  quarter  year  is_weekend
0 2013-01-01            1             1      1        1  2013           0
1 2013-01-02            2             2      1        1  2013           0
2 2013-01-03            3             3      1        1  2013           0
3 2013-01-04            4             4      1        1  2013           0
4 2013-01-05            5             5      1        1  2013           1


,date,day_of_week,day_of_month,month,quarter,year,is_weekend
0,2013-01-01,1,1,1,1,2013,0
1,2013-01-02,2,2,1,1,2013,0
2,2013-01-03,3,3,1,1,2013,0
3,2013-01-04,4,4,1,1,2013,0
4,2013-01-05,5,5,1,1,2013,1


## 6. Promotion Features

The `onpromotion` column represents the count of items in the GROCERY I family on active promotion on that date.

*Note*: Promotions are included as predictive observational features; no causal claim is made regarding promotional impact.

In [7]:
# Inspect promotional feature distribution
print('Promotion summary statistics:')
print(continuous_df['onpromotion'].describe())
print(f"Days with active promotions (>0): {(continuous_df['onpromotion'] > 0).sum()}")
print(f"Days without promotions (=0):    {(continuous_df['onpromotion'] == 0).sum()}")

Promotion summary statistics:
count    1688.000000
mean       17.518957
std        24.726475
min         0.000000
25%         0.000000
50%         6.000000
75%        28.000000
max       167.000000
Name: onpromotion, dtype: float64
Days with active promotions (>0): 1110
Days without promotions (=0):    578


## 7. Multi-Step Forecasting Leakage Analysis & Feature Classification

We evaluate a **28-day forecast horizon**: predicting `2017-07-19` through `2017-08-15` from the forecast origin $T = \text{2017-07-18}$.

### The Multi-Step Leakage Risk
If lag and rolling features are computed simply across the entire continuous series via `sales.shift(1)`:
- On Test Day 1 (July 19): `lag_1` uses actual sales from July 18 ($T$). This is valid historical data.
- On Test Day 2 (July 20): `lag_1` uses actual sales from July 19 ($T+1$). This is **target leakage**! For a 28-day-ahead forecast generated at origin $T$, actual July 19 sales are in the future and would not be known.
- Calculating rolling statistics (e.g., `rolling_mean_7`) across test-period actual sales similarly leaks ground truth into subsequent test days.

### Feature Availability Classification at Forecast Origin ($T$)

| Feature | Type | Available at Origin $T$ for Test Days 1..28? | Multi-Step Forecast Treatment |
| :--- | :--- | :--- | :--- |
| `day_of_week`, `month`, etc. | Calendar | **Yes** (known ex-ante for all 28 days) | Directly usable in test feature matrix |
| `onpromotion` | Promotion | **Yes** (retail promotions are pre-scheduled) | Directly usable in test feature matrix |
| `lag_28` | Target Lag | **Yes** (lookback $28 \ge$ horizon step $k \le 28$, reaches into training set) | Directly usable in test feature matrix |
| `lag_1`, `lag_7`, `lag_14` | Target Lag | **No** (except Day 1; unavailable for later test days without actuals) | Must use **recursive predictions** during inference |
| `rolling_mean_7`, `rolling_mean_28` | Rolling | **No** (except Day 1; unavailable for later test days without actuals) | Must use **recursive predictions** during inference |

**Conclusion**: The simple full-dataset feature table is **not** directly usable for a 28-day-ahead ML forecast without a recursive prediction loop.

## 8. Construction of Leakage-Safe Training and Evaluation Data Structures

We create two distinct data structures:
1. **`continuous_df`**: The full continuous historical series (1,688 days) containing raw actual sales and calendar/promotion attributes. Preserved for inspection and classical time-series models (e.g., ARIMA/ETS) which natively handle multi-step forecasting.
2. **`ml_df`**: The supervised ML modeling dataset:
   - **Training partition** ($t \le T$): Lag and rolling features are fully populated from historical sales.
   - **Test partition** ($t > T$): Day 1 features are populated from history up to $T$. For Days 2..28, features that depend on test-period sales (`lag_1`, `lag_7`, `lag_14`, `rolling_mean_7`, `rolling_mean_28`) are **set to `NaN`** (not populated with future test actuals). During later model evaluation, these will be populated iteratively via recursive forecasting using model predictions.

In [8]:
# Define split boundaries
test_days = 28
test_start_date = continuous_df['date'].max() - pd.Timedelta(days=test_days - 1)
forecast_origin = test_start_date - pd.Timedelta(days=1)

print(f"Forecast origin (last training date): {forecast_origin.strftime('%Y-%m-%d')}")
print(f"Test horizon start date:              {test_start_date.strftime('%Y-%m-%d')}")
print(f"Test horizon end date:                {continuous_df['date'].max().strftime('%Y-%m-%d')}")

# Create ml_df
ml_df = continuous_df.copy()

# Initialize lag and rolling feature columns with NaN
for lag in [1, 7, 14, 28]:
    ml_df[f'lag_{lag}'] = np.nan
ml_df['rolling_mean_7'] = np.nan
ml_df['rolling_mean_28'] = np.nan

# 1. Populate training partition (t <= forecast_origin) using historical sales
train_mask = ml_df['date'] <= forecast_origin
train_sales = ml_df.loc[train_mask, 'sales']

for lag in [1, 7, 14, 28]:
    ml_df.loc[train_mask, f'lag_{lag}'] = train_sales.shift(lag)

ml_df.loc[train_mask, 'rolling_mean_7'] = train_sales.shift(1).rolling(7).mean()
ml_df.loc[train_mask, 'rolling_mean_28'] = train_sales.shift(1).rolling(28).mean()

# 2. Test Day 1 (test_start_date): features are strictly available at forecast origin T
t0_idx = ml_df[ml_df['date'] == test_start_date].index[0]
ml_df.loc[t0_idx, 'lag_1'] = ml_df.loc[t0_idx - 1, 'sales']
ml_df.loc[t0_idx, 'lag_7'] = ml_df.loc[t0_idx - 7, 'sales']
ml_df.loc[t0_idx, 'lag_14'] = ml_df.loc[t0_idx - 14, 'sales']
ml_df.loc[t0_idx, 'lag_28'] = ml_df.loc[t0_idx - 28, 'sales']
ml_df.loc[t0_idx, 'rolling_mean_7'] = ml_df.loc[t0_idx-7:t0_idx-1, 'sales'].mean()
ml_df.loc[t0_idx, 'rolling_mean_28'] = ml_df.loc[t0_idx-28:t0_idx-1, 'sales'].mean()

# 3. For lag_28, lookback is 28 days. Because test horizon is 28 days,
# lag_28 for all test dates falls strictly inside the training set (<= forecast_origin):
test_mask = ml_df['date'] >= test_start_date
for idx in ml_df[test_mask].index:
    lookback_idx = idx - 28
    if lookback_idx <= t0_idx - 1:  # strictly in training partition
        ml_df.loc[idx, 'lag_28'] = ml_df.loc[lookback_idx, 'sales']

# Notice: For Days 2..28 (indices t0_idx + 1 to end),
# lag_1, lag_7, lag_14, rolling_mean_7, rolling_mean_28 remain NaN!
# They will be populated sequentially via recursive predictions, NOT with test actuals.

Forecast origin (last training date): 2017-07-18
Test horizon start date:              2017-07-19
Test horizon end date:                2017-08-15


## 9. Chronological Train / Test Split

We separate the data into training and test partitions based on the forecast origin.

In [9]:
train_df = ml_df[train_mask].copy()
test_df = ml_df[test_mask].copy()

print('=== CHRONOLOGICAL SPLIT SUMMARY ===')
print(f"Training start:  {train_df['date'].min().strftime('%Y-%m-%d')}")
print(f"Training end:    {train_df['date'].max().strftime('%Y-%m-%d')}")
print(f"Training rows:   {len(train_df)}")
print()
print(f"Test start:      {test_df['date'].min().strftime('%Y-%m-%d')}")
print(f"Test end:        {test_df['date'].max().strftime('%Y-%m-%d')}")
print(f"Test rows:       {len(test_df)}")
print(f"Total rows:      {len(train_df) + len(test_df)}")

=== CHRONOLOGICAL SPLIT SUMMARY ===
Training start:  2013-01-01
Training end:    2017-07-18
Training rows:   1660

Test start:      2017-07-19
Test end:        2017-08-15
Test rows:       28
Total rows:      1688


## 10. Strengthened Multi-Step Leakage Validation Checks

We run strict automated verification checks demonstrating that:
1. Test Day 1 features depend strictly on historical data prior to the test start date.
2. Across the entire test horizon (Days 2 to 28), **no feature depends on actual sales from dates $\ge$ test start date**.
3. Actual test-period sales are quarantined strictly as evaluation ground truth targets.

In [10]:
# Check 1: Test Day 1 (July 19) features use only data from t <= forecast_origin (July 18)
day1_row = test_df.iloc[0]
assert day1_row['date'] == test_start_date
assert day1_row['lag_1'] == train_df.loc[train_df['date'] == forecast_origin, 'sales'].values[0]
assert day1_row['lag_7'] == train_df.loc[train_df['date'] == forecast_origin - pd.Timedelta(days=6), 'sales'].values[0]
assert day1_row['rolling_mean_7'] == train_df.loc[train_df['date'] >= forecast_origin - pd.Timedelta(days=6), 'sales'].mean()
print("Check 1 PASSED: Test Day 1 features strictly match forecast origin history.")

# Check 2: Across Days 2..28, verify that lag_1 does NOT use the preceding test day's actual sales
for idx in range(1, len(test_df)):
    prev_actual_test_sales = test_df.iloc[idx - 1]['sales']
    curr_lag1 = test_df.iloc[idx]['lag_1']
    # curr_lag1 must either be NaN (pending recursive prediction) or not equal to the actual unrevealed test sales
    assert np.isnan(curr_lag1), f"LEAKAGE ERROR: Test day {test_df.iloc[idx]['date']} has populated lag_1 using actual test sales!"
print("Check 2 PASSED: No test day after Day 1 has populated lag_1 from actual test sales.")

# Check 3: Verify that lag_7 and lag_14 are NaN once their lookback enters the test period
for idx in range(len(test_df)):
    curr_date = test_df.iloc[idx]['date']
    if curr_date - pd.Timedelta(days=7) >= test_start_date:
        assert np.isnan(test_df.iloc[idx]['lag_7']), f"LEAKAGE: lag_7 at {curr_date} uses test actuals!"
    if curr_date - pd.Timedelta(days=14) >= test_start_date:
        assert np.isnan(test_df.iloc[idx]['lag_14']), f"LEAKAGE: lag_14 at {curr_date} uses test actuals!"
print("Check 3 PASSED: Short lags are not populated once lookback enters the test period.")

# Check 4: Verify rolling means are NaN for Days 2..28
for idx in range(1, len(test_df)):
    assert np.isnan(test_df.iloc[idx]['rolling_mean_7']), f"LEAKAGE: rolling_mean_7 at {test_df.iloc[idx]['date']} uses test actuals!"
    assert np.isnan(test_df.iloc[idx]['rolling_mean_28']), f"LEAKAGE: rolling_mean_28 at {test_df.iloc[idx]['date']} uses test actuals!"
print("Check 4 PASSED: Rolling statistics are not populated using test actuals.")

# Check 5: lag_28 is valid for all test days because 28-day lookback always falls in training period
for idx in range(len(test_df)):
    curr_date = test_df.iloc[idx]['date']
    expected_val = train_df.loc[train_df['date'] == curr_date - pd.Timedelta(days=28), 'sales'].values[0]
    assert test_df.iloc[idx]['lag_28'] == expected_val
print("Check 5 PASSED: lag_28 for all test days is verified against training set actuals.")

print("\nALL STRENGTHENED LEAKAGE ASSERTION CHECKS PASSED SUCCESSFULLY!")

Check 1 PASSED: Test Day 1 features strictly match forecast origin history.
Check 2 PASSED: No test day after Day 1 has populated lag_1 from actual test sales.
Check 3 PASSED: Short lags are not populated once lookback enters the test period.
Check 4 PASSED: Rolling statistics are not populated using test actuals.
Check 5 PASSED: lag_28 for all test days is verified against training set actuals.

ALL STRENGTHENED LEAKAGE ASSERTION CHECKS PASSED SUCCESSFULLY!


## 11. Handling of Initial Burn-in Rows (Lag / Rolling NaNs)

Because the maximum lag and rolling window is 28 days (`lag_28` and `rolling_mean_28`), the first 28 calendar days of the training set (`2013-01-01` to `2013-01-28`) cannot have a complete 28-day history and therefore have `NaN` values.

We do not silently drop these rows from the exported continuous time-series dataset. Instead, we document:
- The continuous dataset preserves all 1,688 rows so classical models (such as ARIMA/ETS) retain an unbroken date index.
- Supervised tabular models (such as XGBoost) will explicitly drop the initial 28 burn-in rows (`df.dropna()` or `df.iloc[28:]`, leaving 1,632 complete training rows) when constructing feature matrix $X$.

In [11]:
# Inspect rows with NaN in lag/rolling features in training partition
train_nan_rows = train_df[train_df['lag_28'].isna()]
print(f"Training burn-in rows with NaN lag_28: {len(train_nan_rows)}")
print(f"Date range of burn-in rows:           {train_nan_rows['date'].min().strftime('%Y-%m-%d')} to {train_nan_rows['date'].max().strftime('%Y-%m-%d')}")
print(f"Complete training feature rows:       {len(train_df) - len(train_nan_rows)}")

Training burn-in rows with NaN lag_28: 28
Date range of burn-in rows:           2013-01-01 to 2013-01-28
Complete training feature rows:       1632


## 12. Save Prepared Modeling Dataset

We save the prepared modeling dataset to `data/processed_store1_grocery1.csv`.

In [12]:
output_path = '../data/processed_store1_grocery1.csv'
ml_df.to_csv(output_path, index=False)

print(f"Derived dataset successfully saved to: {output_path}")
print(f"Final dataset shape: {ml_df.shape}")
print('Columns included:')
for col in ml_df.columns:
    print(f" - {col} ({ml_df[col].dtype})")

Derived dataset successfully saved to: ../data/processed_store1_grocery1.csv
Final dataset shape: (1688, 16)
Columns included:
 - date (datetime64[us])
 - sales (float64)
 - onpromotion (int64)
 - was_missing (int64)
 - day_of_week (int32)
 - day_of_month (int32)
 - month (int32)
 - quarter (int32)
 - year (int32)
 - is_weekend (int64)
 - lag_1 (float64)
 - lag_7 (float64)
 - lag_14 (float64)
 - lag_28 (float64)
 - rolling_mean_7 (float64)
 - rolling_mean_28 (float64)


## Data Preparation Decisions

This section documents the technical decisions made during the data preparation stage:

1. **Missing Christmas Dates & Modeling Assumption**:
   - Exactly four calendar dates were absent from the raw series: December 25 in 2013, 2014, 2015, and 2016.
   - In `holidays_events.csv`, all four coincide with national Christmas Day (`Navidad`).
   - Rather than claiming the raw sales data directly proves store closure, we adopt the explicit modeling assumption that stores were closed on Christmas Day and zero transactions occurred. We therefore reindexed to a continuous daily frequency (`freq='D'`) with `sales = 0.0` and `onpromotion = 0`.
   - An indicator column `was_missing` flags these four imputed closure dates.

2. **Observed Zero-Sales Treatment**:
   - Exactly six zero-sales dates were present in the original dataset: January 1 (New Year's Day) in 2013, 2014, 2015, 2016, and 2017, plus July 7, 2015.
   - These observed zero-sales records are strictly preserved without interpolation, deletion, or conversion to NaN.

3. **Chronological 28-Day Test Split**:
   - The final 28 days (`2017-07-19` to `2017-08-15`, 28 rows) serve as the holdout test set.
   - The training set spans `2013-01-01` to `2017-07-18` (1,660 rows).
   - No random shuffling is used, preserving temporal ordering for realistic forecast evaluation.

4. **Multi-Step Forecasting Leakage Prevention**:
   - For a 28-day-ahead forecast generated at origin $T$ (`2017-07-18`), actual test-period sales for dates $\ge \text{2017-07-19}$ are unobserved.
   - The simple full-dataset shifted feature table is not directly usable for a 28-day-ahead ML forecast because naive shifting would populate Test Days 2..28 with actual ground truth test sales (teacher forcing).
   - For the test horizon, features dependent on test-period sales (`lag_1`, `lag_7`, `lag_14`, `rolling_mean_7`, `rolling_mean_28`) are not populated with actual test sales. In downstream modeling, they will be populated dynamically via recursive forecasting using model predictions.
   - `lag_28` is legitimately available for all 28 test days because its 28-day lookback always reaches into the training partition.

5. **Leakage-Safe Rolling Features**:
   - In the training partition, rolling features are computed strictly on shifted sales: `sales.shift(1).rolling(window).mean()`.
   - Today's sales ($y_t$) is never included in the rolling window calculation for day $t$.
   - Strengthened automated assertions confirmed that no test feature depends on actual sales from dates $\ge$ test start date.